# VERITAS 04 — Instruction tuning (SFT)

**Phase 4 of 16.** Teach the pretrained model to follow instructions — and, for
this project specifically, to **ground, cite, qualify in time, and abstain**.

## Chat format uses real vocabulary tokens

```
<|bos|><|system|>…<|user|>…<|assistant|>…<|eos|>
```

Not the string `"### Assistant:"`, because (a) a string costs 4–6 tokens per
turn, and (b) the model can *generate* a convincing fake role header mid-answer.
A dedicated id cannot be confused with content.

## Assistant-only loss masking

Training on the prompt tokens teaches the model to **generate questions** —
wasted capacity, and it raises the probability of the model continuing with a
fabricated user turn. We zero the loss on system/user tokens and score only the
assistant span (plus `<|eos|>`, so the model learns *when to stop*).

## The four behaviours this stage must install

1. answer strictly from the `<|evidence|>` block;
2. emit `<|claim|>` spans so the verifier can align claims to evidence;
3. emit `<|time|>` qualifiers when a fact is time-bounded;
4. emit `<|unknown|>` when the evidence does not support an answer.

(4) is the important one: **abstention must be trained, not bolted on**.
Otherwise the model always produces a fluent guess that the verifier then has
to delete, and the deletion shows up to the user as an empty answer.

In [ ]:
import sys, os, time, math, json, random
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, torch
torch.manual_seed(1337); np.random.seed(1337); random.seed(1337)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch', torch.__version__)
if DEVICE == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0),
          f'| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
from veritas.tokenizer.bpe import BPETokenizer
from veritas.model.transformer import VeritasLM, ModelConfig
from veritas.train.sft import Turn, SFTDataset, build_example, render, evidence_prompt, load_jsonl
tok = BPETokenizer.load(ROOT/'checkpoints'/'tokenizer.json')
ckpt = ROOT/'checkpoints'/'best.pt'
model = VeritasLM.load(ckpt, DEVICE) if ckpt.exists() else VeritasLM(
    ModelConfig(vocab_size=tok.vocab_size, d_model=384, n_layers=8, n_heads=8,
                n_kv_heads=2, max_seq_len=256)).to(DEVICE)
print('loaded pretrained' if ckpt.exists() else 'NO PRETRAINED CHECKPOINT — run notebook 03 first')

## Build the instruction set

Mix curated examples with *controlled* synthetic ones. Never rely entirely on
generated data: a model trained only on another model's output inherits its
failure modes, including its hallucinations — precisely what this project
exists to prevent.

Each template below teaches one required behaviour.

In [ ]:
# Compact renderer. `evidence_prompt` (used at inference) is verbose because it
# helps a weak model; for SFT the token budget is scarce, and a verbose template
# silently pushes long examples over MAX_LEN where they are DROPPED -- which
# preferentially kills the abstention examples (they carry two evidence blocks)
# and produces a model that never abstains. Measure the drop rate, always.
NL = chr(10)

def ev_prompt(question, evidence):
    lines = ['<|evidence|>[E%d] %s %s' % (i, e['source'], e['date']) + NL + e['text']
             for i, e in enumerate(evidence, 1)]
    lines.append('Q: ' + question)
    return NL.join(lines)

def grounded(question, evidence, answer):
    return [Turn('system', 'Answer only from evidence. Cite [E#]. Abstain if insufficient.'),
            Turn('user', ev_prompt(question, evidence)),
            Turn('assistant', answer)]

examples = []

# 1. grounded answer with a citation
examples.append(grounded(
    'Who is the CEO of Acme Industries?',
    [{'text':'Acme Industries filing: Marcus Lund was appointed chief executive effective February 2026.',
      'source':'sec.gov','date':'2026-02-02','tier':1}],
    '<|claim|>Acme Industries\'s chief executive is Marcus Lund [E1].<|time|>valid from 2026-02-01.'))

# 2. ABSTENTION — evidence about the entity, but not about the attribute
examples.append(grounded(
    "What is Nova Logistics' 2027 revenue guidance?",
    [{'text':'Nova Logistics opened 12 offices in India in 2026.','source':'reuters.com','date':'2026-08-14','tier':2}],
    '<|unknown|>The evidence covers office openings, not 2027 revenue guidance. '
    'I cannot establish this from the available evidence.'))

# 3. CONFLICT — two credible sources disagree; report, do not pick
examples.append(grounded(
    'How many offices did Nova Logistics open in India in 2026?',
    [{'text':'Nova Logistics said it opened 15 new offices in India during 2026.','source':'ir.novalogistics.com','date':'2026-08-01','tier':1},
     {'text':'Nova Logistics opened 12 offices in India in 2026, according to filings.','source':'reuters.com','date':'2026-08-14','tier':2}],
    'Sources disagree: [E1] reports 15 offices, [E2] reports 12. '
    '<|unknown|>The evidence is insufficient to establish the exact number.'))

# 4. STALE EVIDENCE — answer, but qualify the validity window
examples.append(grounded(
    'What is the current status of the Meridian Port expansion?',
    [{'text':'The Meridian Port expansion is in the planning phase, with approval expected in 2024.','source':'gov.example.gov','date':'2023-05-01','tier':1}],
    '<|claim|>The Meridian Port expansion was in the planning phase [E1].'
    '<|time|>valid as of May 2023. No more recent evidence confirms the current status.'))

# 5. HISTORICAL vs CURRENT — the model must honour the tense of the question
examples.append(grounded(
    'Who was the CEO of Acme Industries in 2024?',
    [{'text':'Acme Industries annual report 2024. The chief executive is Dana Whitfield.','source':'ir.acme.com','date':'2024-03-01','tier':1},
     {'text':'Acme Industries filing: Marcus Lund was appointed chief executive effective February 2026.','source':'sec.gov','date':'2026-02-02','tier':1}],
    '<|claim|>In 2024 the chief executive of Acme Industries was Dana Whitfield [E1].'
    '<|time|>valid 2024. The current chief executive is Marcus Lund [E2].'))

print(f'{len(examples)} curated templates')
print('\n--- rendered example ---')
print(render(examples[1])[:400])

### Scale it with controlled substitution

Only names, dates and values vary; the *behaviour* being taught stays fixed.
That buys volume without importing a generator model's errors.

In [ ]:
firms = ['Cobalt Works','Vireo Health','Northwind Rail','Stellar Foods','Kestrel Bank','Orion Systems']
people = ['Alicia Moreau','Tomas Berg','Hana Suzuki','Emeka Obi','Lena Petrov','Raj Malhotra']
rng = random.Random(0)
synthetic = []
for _ in range(400):
    f, p = rng.choice(firms), rng.choice(people)
    yr = rng.choice([2023,2024,2025,2026])
    kind = rng.random()
    if kind < 0.45:
        synthetic.append(grounded(f'Who is the CEO of {f}?',
            [{'text':f'{f} filing: {p} was appointed chief executive effective January {yr}.',
              'source':'sec.gov','date':f'{yr}-01-15','tier':1}],
            f'<|claim|>{f}\'s chief executive is {p} [E1].<|time|>valid from January {yr}.'))
    elif kind < 0.75:
        synthetic.append(grounded(f'What is {f}\'s current headcount?',
            [{'text':f'{f} filing: {p} was appointed chief executive effective January {yr}.',
              'source':'sec.gov','date':f'{yr}-01-15','tier':1}],
            '<|unknown|>The evidence does not state headcount. I cannot establish this.'))
    else:
        a, b = rng.randint(10,40), rng.randint(10,40)
        synthetic.append(grounded(f'How many sites does {f} operate?',
            [{'text':f'{f} said it operates {a} sites.','source':f'ir.{f.split()[0].lower()}.com','date':f'{yr}-06-01','tier':1},
             {'text':f'{f} operates {b} sites, according to filings.','source':'reuters.com','date':f'{yr}-06-10','tier':2}],
            f'Sources disagree: [E1] reports {a}, [E2] reports {b}. '
            f'<|unknown|>The evidence is insufficient to establish the exact number.'))

conversations = examples*20 + synthetic
rng.shuffle(conversations)
print(f'{len(conversations)} conversations '
      f'({sum("<|unknown|>" in t[-1].content for t in conversations)} teach abstention)')

## Encode with the loss mask

Inspect the mask before training. Green (1) positions contribute to the loss;
everything in the prompt must be 0.

In [ ]:
MAX_LEN = model.cfg.max_seq_len    # must match what the model was pretrained with
encoded, kept_abstain, total_abstain = [], 0, 0
for c in conversations:
    is_abstain = '<|unknown|>' in c[-1].content
    total_abstain += is_abstain
    e = build_example(tok, c, MAX_LEN)
    if e:
        encoded.append(e); kept_abstain += is_abstain

fit = len(encoded)/len(conversations)
print(f'{len(encoded)}/{len(conversations)} fit in {MAX_LEN} tokens  ({fit:.0%})')
print(f'abstention examples kept: {kept_abstain}/{total_abstain} '
      f'= {kept_abstain/max(1,len(encoded)):.0%} of the training set')
# Length filtering must not silently rebalance the classes. If abstention
# examples drop below ~20% the model will not learn to abstain at all.
assert fit > 0.6, 'too many examples dropped -- shorten the template or raise MAX_LEN'
assert kept_abstain/max(1,len(encoded)) > 0.2, 'abstention examples were filtered away'

ex = encoded[0]
toks = [tok.decode([i]) for i in ex['ids']]
print('\nfirst 40 positions (mask | token):')
print(' '.join(f"{m}:{t.strip()[:10] or '·'}" for m,t in list(zip(ex['mask'], toks))[:40]))
frac = sum(e['mask'].count(1) for e in encoded)/sum(len(e['ids']) for e in encoded)
print(f'\n{frac:.1%} of positions are scored — the rest is prompt, correctly masked out')

In [ ]:
ds = SFTDataset(encoded, tok.special_tokens['<|pad|>'], MAX_LEN)
opt = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01, betas=(0.9,0.95))
# LR is ~6x lower than pretraining: SFT adapts an existing model, and a high LR
# causes catastrophic forgetting of everything pretraining bought.
STEPS, BS = 600, 8
losses = []
model.train()
for step in range(STEPS):
    x, y, m = ds.batch(BS, DEVICE)
    with torch.autocast(device_type=DEVICE.split(':')[0], dtype=torch.bfloat16, enabled=DEVICE=='cuda'):
        _, loss = model(x, y, loss_mask=m)
    opt.zero_grad(set_to_none=True); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    losses.append(loss.item())
    if step % 40 == 0: print(f'step {step:4d} | masked loss {loss.item():.4f}')
import matplotlib.pyplot as plt
plt.figure(figsize=(7,2.5)); plt.plot(losses); plt.xlabel('step')
plt.ylabel('assistant-only loss'); plt.grid(alpha=.3); plt.show()

## Test the trained behaviours

The question is not "is the prose good" — at 30M parameters it will not be.
The question is **does it abstain when it should**. Compare an answerable
prompt with an unanswerable one.

In [ ]:
def ask(question, evidence, max_new=70):
    prompt = ('<|bos|><|system|>Answer only from evidence. Cite [E#]. Abstain if '
              'insufficient.<|user|>' + ev_prompt(question, evidence) + '<|assistant|>')
    ids = torch.tensor([tok.encode(prompt)], device=DEVICE)
    model.eval()
    with torch.inference_mode():
        out = model.generate(ids, max_new_tokens=max_new, temperature=0.2, top_k=30,
                             eos_id=tok.special_tokens['<|eos|>'])
    return tok.decode(out[0, ids.shape[1]:].tolist())

ev = [{'text':'Kestrel Bank filing: Lena Petrov was appointed chief executive effective January 2026.',
       'source':'sec.gov','date':'2026-01-15','tier':1}]
print('ANSWERABLE:\n ', ask('Who is the CEO of Kestrel Bank?', ev), '\n')
print('NOT ANSWERABLE FROM THIS EVIDENCE:\n ', ask("What is Kestrel Bank's current headcount?", ev))

In [ ]:
unk = tok.special_tokens['<|unknown|>']
answerable = sum(unk in tok.encode(ask('Who is the CEO of Kestrel Bank?', ev)) for _ in range(5))
unanswerable = sum(unk in tok.encode(ask("What is Kestrel Bank's headcount?", ev)) for _ in range(5))
print(f'abstained on ANSWERABLE   : {answerable}/5   (want 0 — over-abstention is also a failure)')
print(f'abstained on UNANSWERABLE : {unanswerable}/5 (want 5)')
print('\nThis pair IS the abstention-quality metric. Both directions matter:')
print('a model that always refuses scores perfectly on one and uselessly on the other.')
model.save(str(ROOT/'checkpoints'/'sft.pt')); print('\nsaved -> checkpoints/sft.pt')

Next: **05 — retrieval.**